# 🎯 6. Memory in AI Agents

Without memory, an agent forgets everything between turns — like talking to someone with amnesia. Memory is what makes agents *useful* across interactions.

In this notebook:

1. **Memory taxonomy** — working, short-term, long-term, episodic
2. **Conversation memory** — maintaining chat history
3. **LangGraph state persistence** — checkpointing with SQLite
4. **Memory summarization** — compressing long histories
5. **Context engineering** — managing the context window
6. **Context failure modes** — poisoning, distraction, confusion, clash

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
# if not os.environ.get("OPENAI_API_KEY"):
#     import getpass
#     os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

LLM_MODEL   = "gpt-4o-mini"

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
print(f'Model: {LLM_MODEL}')

Model: gpt-4o-mini


## 6.1 Memory Taxonomy

AI agents use different types of memory for different purposes:

| Type | Scope | Duration | Purpose | Example |
|------|-------|----------|---------|--------|
| **Working** | Current step | Milliseconds | Active computation | Current tool arguments |
| **Short-term** | Single session | Minutes | Conversation context | Chat history |
| **Long-term** | Cross-session | Permanent | User preferences, facts | "User prefers Python" |
| **Episodic** | Historical | Permanent | Past experiences | "Last time search failed, used different query" |
| **Entity** | Extracted | Dynamic | Structured entities | "John: Manager at Acme Corp" |

In practice:
- **Short-term** = the messages list in the agent loop
- **Long-term** = persisted to a database or file
- **Episodic** = learning from past interactions

## 6.2 Conversation Memory — The Messages List

The simplest form of memory is the conversation history itself:

In [2]:
class ConversationMemory:
    """Simple conversation memory with a sliding window"""

    def __init__(self, max_messages: int = 20):
        self.messages = []
        self.max_messages = max_messages
    
    def add(self, role: str, content: str):
        self.messages.append({'role': role, 'content':content})
        # sliding window: keep system + last N messages
        if len(self.messages) > self.max_messages:
            system_msgs = [m for m in self.messages if m['role'] == 'system']
            other_msgs = [m for m in self.messages if m['role'] != 'system']
            self.messages = system_msgs + other_msgs[-(self.max_messages - len(system_msgs)):]
    
    def get_messages(self) -> list:
        return self.messages.copy()
    
    def clear(self):
        self.messages = []

# Demo conversation
memory = ConversationMemory(max_messages=10)
memory.add('system', "You are a helpful assistan")
memory.add('user', "My name is Alex")
memory.add("assistant", "Nice to meet you, Alex!")
memory.add("user", "I work at a statup called Datanous")
memory.add("assistant", "That sounds grate! What does Datanous do?")

response = llm.invoke([
    *[HumanMessage(content=m['content']) if m['role'] == 'user' 
      else SystemMessage(content=m['content']) if m['role'] == 'system'
      else AIMessage(content=m['content'])
      for m in memory.get_messages()],
    HumanMessage(content='What is my name and where do I work?')
])
print(f'Agent: {response.content}')
print(f'Messages in memory: {len(memory.messages)}')

Agent: Your name is Alex, and you work at a startup called Datanous.
Messages in memory: 5


## 6.3 LangGraph State Persistence — Checkpointing

LangGraph can persist state across runs using **checkpointers**. This means:
- Agent state survives between sessions
- You can resume a conversation after closing the notebook
- Multiple threads (conversations) are supported simultaneously

<img src="images/langgraph-checkpoint.png" width="50%" style="border-radius:10px;margin:12px 0;"/>

In [3]:
# create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent

memory = MemorySaver()

app = create_agent(
    model=ChatOpenAI(model=LLM_MODEL, temperature=0),
    tools=[], # no tools for the moment
    system_prompt="You are a helpful assistant",
    checkpointer=memory
)

In [5]:
config = {'configurable': {"thread_id": "user-alex-001"}}

r1 = app.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Alex and i love Python!")]},
    config=config
)

print(f"Turn 1: {r1['messages'][-1].content}")

r2 = app.invoke(
     {"messages": [HumanMessage(content="What is my name and what language do i like?")]},
    config=config
)

print(f"Turn 2: {r2['messages'][-1].content}")


Turn 1: Hi Alex! It's great to hear that you love Python! It's a versatile and powerful programming language. What do you enjoy most about it? Are you working on any specific projects or learning something new?
Turn 2: Your name is Alex, and you love Python!


In [6]:
config2 = {'configurable': {'thread_id': 'user-other-002'}}
r3 = app.invoke(
    {'messages': [HumanMessage(content='Do you remember my name?')]},
    config=config2,
)
print(f'New thread: {r3["messages"][-1].content}')

New thread: I don’t have the ability to remember personal information or previous interactions, including your name. How can I assist you today?


### 6.3.1 A better and more advance implementation with InMemory Saver

In [7]:
import os
import operator
from typing_extensions import TypedDict, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

In [9]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    turn_count: Annotated[int, operator.add]

def chat_node(state: ChatState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are helpful assistant. Be concise."),
        *state['messages'],
    ])
    return {
        "messages": [response],
        "turn_count": 1
    }

In [10]:
graph = StateGraph(ChatState)

graph.add_node("chat", chat_node)

graph.add_edge(START, "chat")
graph.add_edge("chat", END)

# Checkpointer
# InMemorySave -> RAM
# SQLite ... (soon)
checkpointer = InMemorySaver()
app = graph.compile(checkpointer=checkpointer)

In [11]:
def chat(thread_id: str, user_message: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke(
        {
            "messages": [HumanMessage(content=user_message)],
            "turn_count": 0,
        },
        config=config,
    )
    reply = result["messages"][-1].content
    turns = result["turn_count"]
    print(f"[thread={thread_id} | turn #{turns}]")
    print(f"  User : {user_message}")
    print(f"  Agent: {reply}")
    return reply

In [12]:
# Demo
print("=" * 55)
print("ALEX — Turn 1: εισαγωγή")
print("=" * 55)
chat("user-alex-001", "Hi! My name is Alex and I love Python.")
print()
print("=" * 55)
print("ALEX — Turn 2: ο agent θυμάται το όνομα & το hobby")
print("=" * 55)
chat("user-alex-001", "What is my name and what language do I like?")
print()

print("=" * 55)
print("MARIA — ξεχωριστό thread → μηδενική μνήμη από τον Alex")
print("=" * 55)
chat("user-maria-001", "What is my name and what language do I like?")

ALEX — Turn 1: εισαγωγή
[thread=user-alex-001 | turn #1]
  User : Hi! My name is Alex and I love Python.
  Agent: Hi Alex! That's great to hear! Python is a versatile and powerful programming language. What do you enjoy most about it?

ALEX — Turn 2: ο agent θυμάται το όνομα & το hobby
[thread=user-alex-001 | turn #2]
  User : What is my name and what language do I like?
  Agent: Your name is Alex, and you love Python.

MARIA — ξεχωριστό thread → μηδενική μνήμη από τον Alex
[thread=user-maria-001 | turn #1]
  User : What is my name and what language do I like?
  Agent: I'm sorry, but I don't have access to personal information about you, including your name or language preferences.


"I'm sorry, but I don't have access to personal information about you, including your name or language preferences."

6.3.2. SQLite implementation - Persistence Memory

In [13]:
from langgraph.checkpoint.sqlite import SqliteSaver

DB_PATH = "checkpoints.sqlite"

with SqliteSaver.from_conn_string(DB_PATH) as sqlite_checkpointer:
    sqlite_app = graph.compile(checkpointer=sqlite_checkpointer)
    config = {"configurable": {"thread_id": "user-alex-sqlite"}}

    # 1st execution: insert
    sqlite_app.invoke(
        {
            "messages": [HumanMessage(content="My name is Alex and i love Python!")],
            "turn_count": 0
        },
        config=config
    )

    # Second execution: agent remembers...?
    result = sqlite_app.invoke(
        {
            "messages": [HumanMessage(content="What do you know about me?")],
            "turn_count":0
        },
        config=config
    )


In [14]:
print(result['messages'][-1].content)
print(f"\nState saved in: {DB_PATH}")

I only know that your name is Alex and that you love Python. I don’t have any personal information about you beyond that. How can I assist you today?

State saved in: checkpoints.sqlite


In [15]:
# Verification: Simulate a kernel restart
# Φτιάχνουμε ΝΕΑ σύνδεση στο ίδιο αρχείο (καμία shared μεταβλητή από πριν).
# Αν η μνήμη επιβίωσε, ο agent θα ξέρει ότι μιλάμε για τον Alex.

with SqliteSaver.from_conn_string(DB_PATH) as fresh_checkpointer:
    fresh_app = graph.compile(checkpointer=fresh_checkpointer)

    config = {"configurable": {"thread_id": "user-alex-sqlite"}}

    # 1. Διαβάζουμε το checkpoint απευθείας (χωρίς νέο invoke)
    snapshot = fresh_app.get_state(config)

    print("=" * 55)
    print("📸 CHECKPOINT SNAPSHOT (raw state from SQLite)")
    print("=" * 55)
    print(f"Turn count : {snapshot.values.get('turn_count')}")
    print(f"Messages   : {len(snapshot.values.get('messages', []))}")
    print()

    for msg in snapshot.values.get("messages", []):
        role = type(msg).__name__.replace("Message", "").upper()
        print(f"  [{role}] {msg.content}")

    # 2. Νέο ερώτημα στο ίδιο thread — ο agent συνεχίζει κανονικά
    print()
    print("=" * 55)
    print("💬 NEW TURN (same thread, fresh connection)")
    print("=" * 55)

    result = fresh_app.invoke(
        {
            "messages": [HumanMessage(content="What programming language do I like?")],
            "turn_count": 0,
        },
        config=config,
    )

    print("What programming language do I like?")
    print(f"Agent: {result['messages'][-1].content}")
    print(f"Total turns so far: {result['turn_count']}")


📸 CHECKPOINT SNAPSHOT (raw state from SQLite)
Turn count : 2
Messages   : 4

  [HUMAN] My name is Alex and i love Python!
  [AI] Hi Alex! That's great to hear! Python is a versatile and powerful programming language. What do you enjoy most about it?
  [HUMAN] What do you know about me?
  [AI] I only know that your name is Alex and that you love Python. I don’t have any personal information about you beyond that. How can I assist you today?

💬 NEW TURN (same thread, fresh connection)
What programming language do I like?
Agent: You like Python!
Total turns so far: 3


## 6.4 Memory Summarization

As conversations grow, the context window fills up. **Summarization** compresses old messages while preserving key information:

<img src="images/memory-summarization.png" width="70%" style="border-radius:10px;margin:12px 0;"/>

In [16]:
def summarize_history(messages: list, keep_recent: int = 4) -> list:
    """Summarize older messages, keep recent ones intact"""
    if len(messages) <= keep_recent + 1:
        return messages
    
    # separate system and convesation messages
    system_msgs = [m for m in messages if isinstance(m, SystemMessage)]
    conv_msgs = [m for m in messages if not isinstance(m, SystemMessage)]

    # Split int old and recent
    old_msgs = conv_msgs[:-keep_recent]
    recent_msgs = conv_msgs[-keep_recent:]

    old_text = "\n".join([f"{m.__class__.__name__}: {m.content}" for m in old_msgs])

    summary = llm.invoke([
        SystemMessage(content="Summarize this conversation history. Keep key facts, name and decisions."),
        HumanMessage(content=old_text)
    ])

    summary_msg = SystemMessage(content=f"Previous conversation summary: {summary.content}")
    return system_msgs + [summary_msg] + recent_msgs


In [17]:
long_history = [
    SystemMessage(content='You are a travel planning assistant.'),
    HumanMessage(content='I want to visit Japan in March.'),
    AIMessage(content='March is cherry blossom season! Tokyo and Kyoto are popular.'),
    HumanMessage(content='I prefer Tokyo. Budget is $3000.'),
    AIMessage(content='With $3000 in Tokyo, I recommend staying in Shinjuku for 7 days.'),
    HumanMessage(content='What about food?'),
    AIMessage(content='Budget about $50/day for food. Try ramen, sushi, and izakaya.'),
    HumanMessage(content='Any must-see attractions?'),
    AIMessage(content='Senso-ji, Meiji Shrine, Shibuya Crossing, and Akihabara.'),
]

print(f'Before: {len(long_history)} messages')
summarized = summarize_history(long_history, keep_recent=4)
print(f'After:  {len(summarized)} messages')
print(f'\nSummary: {summarized[1].content}')

Before: 9 messages
After:  6 messages

Summary: Previous conversation summary: The conversation involves a user expressing a desire to visit Japan in March, which is noted as cherry blossom season. The user prefers Tokyo and has a budget of $3000. The AI recommends staying in Shinjuku for 7 days within that budget.


In [18]:
from IPython.display import display, Markdown

def visualize_history(messages: list, title: str) -> None:
    md = []

    md.append(f"## {title}")
    md.append("---")

    for index, message in enumerate(messages):
        message_type = message.__class__.__name__

        if isinstance(message, SystemMessage):
            label = "SYSTEM"
        elif isinstance(message, HumanMessage):
            label = "USER"
        elif isinstance(message, AIMessage):
            label = "ASSISTANT"
        else:
            label = "UNKNOWN"

        content = str(message.content)

        # Indent content so Markdown preserves it as a code-style block
        formatted_content = "\n".join(
            f"    {line}" for line in content.splitlines()
        )

        md.append(f"### [{index}] {label} `({message_type})`")
        md.append(formatted_content if formatted_content else "    ")
        md.append("---")

    display(Markdown("\n\n".join(md)))


visualize_history(long_history, "BEFORE SUMMARIZATION")
visualize_history(summarized, "AFTER SUMMARIZATION")

## BEFORE SUMMARIZATION

---

### [0] SYSTEM `(SystemMessage)`

    You are a travel planning assistant.

---

### [1] USER `(HumanMessage)`

    I want to visit Japan in March.

---

### [2] ASSISTANT `(AIMessage)`

    March is cherry blossom season! Tokyo and Kyoto are popular.

---

### [3] USER `(HumanMessage)`

    I prefer Tokyo. Budget is $3000.

---

### [4] ASSISTANT `(AIMessage)`

    With $3000 in Tokyo, I recommend staying in Shinjuku for 7 days.

---

### [5] USER `(HumanMessage)`

    What about food?

---

### [6] ASSISTANT `(AIMessage)`

    Budget about $50/day for food. Try ramen, sushi, and izakaya.

---

### [7] USER `(HumanMessage)`

    Any must-see attractions?

---

### [8] ASSISTANT `(AIMessage)`

    Senso-ji, Meiji Shrine, Shibuya Crossing, and Akihabara.

---

## AFTER SUMMARIZATION

---

### [0] SYSTEM `(SystemMessage)`

    You are a travel planning assistant.

---

### [1] SYSTEM `(SystemMessage)`

    Previous conversation summary: The conversation involves a user expressing a desire to visit Japan in March, which is noted as cherry blossom season. The user prefers Tokyo and has a budget of $3000. The AI recommends staying in Shinjuku for 7 days within that budget.

---

### [2] USER `(HumanMessage)`

    What about food?

---

### [3] ASSISTANT `(AIMessage)`

    Budget about $50/day for food. Try ramen, sushi, and izakaya.

---

### [4] USER `(HumanMessage)`

    Any must-see attractions?

---

### [5] ASSISTANT `(AIMessage)`

    Senso-ji, Meiji Shrine, Shibuya Crossing, and Akihabara.

---

## 6.5 Context Engineering — Failure Modes

Context management is critical. If done poorly, the agent fails in subtle ways:

| Failure Mode | Cause | Symptom | Fix |
|-------------|-------|---------|-----|
| **Poisoning** | Hallucination stored as memory | Agent believes false facts | Validate before storing |
| **Distraction** | Oversized history | Agent ignores recent context | Periodic summarization |
| **Confusion** | Too many tools in context | Agent picks wrong tool | Limit to < 30 tools, use RAG for tool selection |
| **Clash** | Contradictory instructions | Unpredictable behavior | Prune contradictions, use scratchpad |

> **Key Insight**: Context engineering is NOT the same as prompt engineering.
> - Prompt engineering = static instructions
> - Context engineering = dynamic information management across sessions

## 💡 Exercise 6: Build an Agent with Persistent Memory

**Task**: Build a LangGraph agent that:
1. Remembers user preferences across turns (use MemorySaver)
2. Summarizes history when it exceeds 10 messages
3. Supports multiple threads (different users)

Test by having a multi-turn conversation about travel planning, then asking the agent to recall earlier preferences.

In [ ]:
# Exercise 6: YOUR CODE HERE


## 📝 Summary

| Concept | Key Takeaway |
|---------|-------------|
| **Memory types** | Working → Short-term → Long-term → Episodic |
| **Conversation memory** | Messages list with sliding window |
| **Checkpointing** | LangGraph MemorySaver persists state across runs |
| **Summarization** | Compress old messages while keeping key facts |
| **Context failures** | Poisoning, distraction, confusion, clash — all preventable |

### What's Next

In **Notebook 07: RAG-Powered Agents**, we connect agents to knowledge bases — giving them access to documents, data, and domain expertise through retrieval.